<a href="https://colab.research.google.com/github/ycui4407/cs250-fall2026-Yi-Cui/blob/main/Labs/L03_Cui_Yi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 250 — Lab 3
## Arithmetic Expressions, Booleans & Program Logic

San Francisco Bay University · Fall 2026 ·
Week 2, Session 2 · 50 minutes

---

### What you will practise
1. Using all seven arithmetic operators, especially `//` and `%`
2. Applying operator precedence and parenthesising for clarity
3. Translating a mathematical formula into a Python expression
4. Producing Boolean values with comparison and logical operators
5. Turning a flowchart into working code — and code back into a flowchart
6. Structuring a program as Input → Process → Output

### Before you start
You should have completed **Lab 2**. This lab assumes you are comfortable with
variables, `input()`, type conversion and f-strings.

### How this lab checks your work
Every task is followed by a **CHECK** cell. Run it after you finish the task:

| | |
|---|---|
| 🟢 **PASSED** | that task is done and correct |
| 🔴 **NOT YET** | something is wrong — the cell tells you *what* and *why* |
| 🟡 **TO DO** | you have not filled that task in yet |

The very last cell of the notebook, `final_check()`, re-runs **all** the checkpoints
and shows you a dashboard. When every light is green it will tell you exactly how to
submit.

### Submitting
When all checkpoints are green, name the notebook **`L03_firstname_lastname.ipynb`**,
push it to GitHub, and submit the notebook's GitHub URL to Canvas.

---
## Part 0 · Setup

### ▶️ Run this cell first

It loads the checkpoint helpers used by every CHECK cell in this lab. You do not need
to understand it yet, and you should not change it. If you ever restart the kernel,
run it again (or just use **Runtime → Restart and run all**).

In [1]:
# =====================================================================
#  CS 250 - Lab 3 - CHECKPOINT SETUP.   RUN THIS CELL FIRST.
#
#  This cell defines report() and final_check(), which the CHECK cells
#  and the last cell of the notebook use. You are not expected to
#  understand this code yet - just run it, and do not edit it.
# =====================================================================
import re

PASS, FAIL, TODO = "pass", "fail", "todo"
MISSING = "<no such variable>"
CHECKS, ORDER = {}, []


def _checkpoint(cid, title):
    """Register a grading function under a checkpoint id."""
    def wrap(fn):
        CHECKS[cid] = (title, fn)
        if cid not in ORDER:
            ORDER.append(cid)
        return fn
    return wrap


def _v(name):
    """The current value of a notebook variable, or MISSING."""
    return globals().get(name, MISSING)


def _num(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def _todo(*names):
    """Names that do not exist yet or are still None."""
    return [n for n in names if _v(n) is MISSING or _v(n) is None]


def _close(x, y, tol=1e-6):
    return _num(x) and abs(x - y) <= tol


def _sources():
    """Source of every cell run so far (used for a few soft hints)."""
    try:
        return [str(s) for s in globals().get("In", [])]
    except Exception:
        return []


# ---------------------------------------------------------------- 0.1
@_checkpoint("0.1", "Part 0 - your details")
def _c01():
    name = _v("student_name")
    if name is MISSING:
        return TODO, ["The Part 0 cell has not been run yet."]
    if not isinstance(name, str) or not name.strip():
        return FAIL, ['student_name must be a non-empty piece of text, e.g. "Ada Lovelace".']
    if "your name" in name.lower():
        return FAIL, ['student_name is still the placeholder "Your Name Here".',
                      "Replace it with your real first and last name - the submission",
                      "filename is built from it."]
    if len(name.split()) < 2:
        return FAIL, ["Please give both a first and a last name.",
                      "Canvas expects the file to be named L03_firstname_lastname."]
    date = _v("lab_date")
    if not isinstance(date, str) or not date.strip():
        return FAIL, ['lab_date should be a piece of text such as "2026-09-03".']
    return PASS, ["Hello " + name.strip() + " - your details are recorded."]


# ---------------------------------------------------------------- 1.3
@_checkpoint("1.3", "Task 1.3 - make change")
def _c13():
    left = _todo("quarters", "dimes", "nickels", "pennies")
    if left:
        return TODO, ["Still None or missing: " + ", ".join(left) + "."]

    got = [_v(n) for n in ("quarters", "dimes", "nickels", "pennies")]
    if not all(_num(x) for x in got):
        return FAIL, ["Every coin count must be a number."]

    cents = _v("total_cents")
    if not _num(cents):
        cents = 287

    floats = [n for n, x in zip(("quarters", "dimes", "nickels", "pennies"), got)
              if isinstance(x, float)]
    if floats:
        return FAIL, ["These are floats, but you cannot hand out part of a coin: "
                      + ", ".join(floats) + ".",
                      "Use // (floor division), not / (true division)."]

    q = cents // 25
    d = (cents % 25) // 10
    n = (cents % 25 % 10) // 5
    p = cents % 25 % 10 % 5
    want = [q, d, n, p]
    if got == want:
        def _plural(count, one, many):
            return "%d %s" % (count, one if count == 1 else many)
        return PASS, ["%d cents = %s, %s, %s, %s."
                      % (cents, _plural(q, "quarter", "quarters"),
                         _plural(d, "dime", "dimes"),
                         _plural(n, "nickel", "nickels"),
                         _plural(p, "penny", "pennies"))]

    msg = []
    for label, g, w, hint in zip(("quarters", "dimes", "nickels", "pennies"), got, want,
                                 ("total_cents // 25",
                                  "the leftover after quarters, // 10",
                                  "the leftover after dimes, // 5",
                                  "the leftover after nickels")):
        if g != w:
            msg.append("%-9s got %s, expected %s   (%s)" % (label + ":", g, w, hint))
    paid = 25 * got[0] + 10 * got[1] + 5 * got[2] + got[3]
    if paid != cents:
        msg.append("Your coins are worth %d cents but the customer paid %d." % (paid, cents))
        msg.append("After taking out a coin you must keep working on the REMAINDER (%),")
        msg.append("not on the original total.")
    else:
        msg.append("The values add up to the right amount, but you did not always take")
        msg.append("the largest coin first.")
    return FAIL, msg


# ---------------------------------------------------------------- 2.1
_P21 = [
    ("p1", "2 + 3 * 4", 2 + 3 * 4,
     "* binds tighter than +, so the multiplication happens first."),
    ("p2", "(2 + 3) * 4", (2 + 3) * 4,
     "Parentheses always win, whatever is inside them goes first."),
    ("p3", "10 - 2 - 3", 10 - 2 - 3,
     "- is left-associative: (10 - 2) - 3, not 10 - (2 - 3)."),
    ("p4", "2 ** 3 ** 2", 2 ** 3 ** 2,
     "** is the odd one out - it is RIGHT-associative: 2 ** (3 ** 2) = 2 ** 9."),
    ("p5", "20 // 3 * 3", 20 // 3 * 3,
     "// and * have equal precedence, so it runs left to right: (20 // 3) * 3 = 6 * 3."),
    ("p6", "7 + 15 % 4", 7 + 15 % 4,
     "% binds tighter than +: 7 + (15 % 4) = 7 + 3."),
]


@_checkpoint("2.1", "Task 2.1 - precedence predictions")
def _c21():
    left = _todo(*[k for k, _, _, _ in _P21])
    if left:
        return TODO, ["Not predicted yet: " + ", ".join(left) + "."]

    wrong = []
    for key, expr, want, hint in _P21:
        got = _v(key)
        if not _num(got):
            wrong.append("%s should be a number, not %r." % (key, got))
        elif got != want:
            wrong.append("%s   %-14s you said %s, it is %s" % (key, expr, got, want))
            wrong.append("      " + hint)
    if wrong:
        return FAIL, wrong
    return PASS, ["All six predictions match - you have the precedence rules down."]


# ---------------------------------------------------------------- 2.2
@_checkpoint("2.2", "Task 2.2 - rewrite for clarity")
def _c22():
    if _todo("average"):
        return TODO, ["average has not been computed yet."]
    avg = _v("average")
    if not _num(avg):
        return FAIL, ["average should be a number, not %r." % (avg,)]

    a, b, c = _v("a"), _v("b"), _v("c")
    if not all(_num(x) for x in (a, b, c)):
        a, b, c = 88, 92, 79
    want = (a + b + c) / 3
    if _close(avg, want, 0.005):
        return PASS, ["Average is %.2f - the parentheses are in the right place." % avg]

    msg = ["Got %.4f, expected %.4f." % (avg, want)]
    if _close(avg, a + b + c / 3, 0.005):
        msg.append("That is the UNPARENTHESISED version: only c was divided by 3.")
        msg.append("A fraction bar is a big parenthesis - wrap the whole numerator.")
    elif _close(avg, (a + b + c) / 2, 0.005):
        msg.append("You divided by 2. There are three scores.")
    elif _close(avg, a + b + c, 0.005):
        msg.append("You added the scores but never divided by 3.")
    else:
        msg.append("Write it as (a + b + c) / 3.")
    return FAIL, msg


# ---------------------------------------------------------------- 3.2
@_checkpoint("3.2", "Task 3.2 - body mass index")
def _c32():
    if _todo("bmi"):
        return TODO, ["bmi has not been computed yet."]
    got = _v("bmi")
    if not _num(got):
        return FAIL, ["bmi should be a number, not %r." % (got,)]

    w, h = _v("weight_kg"), _v("height_m")
    if not (_num(w) and _num(h) and h):
        w, h = 68.0, 1.72
    want = w / (h * h)
    if _close(got, want, 0.01):
        return PASS, ["BMI is %.2f - correct." % got]

    msg = ["Got %.4f, expected about %.2f." % (got, want)]
    if _close(got, w / h * h, 0.01):
        msg.append("Dividing by the height and then multiplying by it cancels out,")
        msg.append("so you got the weight back. Parenthesise the whole denominator:")
        msg.append("weight_kg / (height_m * height_m), or use height_m ** 2.")
    elif _close(got, w / h, 0.01):
        msg.append("You divided by the height only once - the formula squares it.")
    elif _close(got, w * h * h, 0.01):
        msg.append("You multiplied where the formula divides.")
    else:
        msg.append("The height goes in the denominator, twice.")
    return FAIL, msg


# ---------------------------------------------------------------- 3.3
@_checkpoint("3.3", "Task 3.3 - distance between two points")
def _c33():
    if _todo("distance"):
        return TODO, ["distance has not been computed yet."]
    got = _v("distance")
    if not _num(got):
        return FAIL, ["distance should be a number, not %r." % (got,)]

    x1, y1, x2, y2 = _v("x1"), _v("y1"), _v("x2"), _v("y2")
    if not all(_num(v) for v in (x1, y1, x2, y2)):
        x1, y1, x2, y2 = 1, 2, 4, 6
    dx, dy = x2 - x1, y2 - y1
    want = (dx ** 2 + dy ** 2) ** 0.5
    if _close(got, want, 1e-9):
        return PASS, ["Distance is %s - correct." % got]

    msg = ["Got %s, expected %s." % (got, want)]
    if _close(got, dx ** 2 + dy ** 2, 1e-9):
        msg.append("That is the sum of the squares - you never took the square root.")
        msg.append("Raise the whole sum to the power 0.5.")
    elif _close(got, dx ** 2 + dy ** 2 ** 0.5, 1e-9):
        msg.append("Precedence bug: ** binds tighter than +, so only the last term")
        msg.append("got the square root. Wrap the WHOLE sum in parentheses before ** 0.5.")
    elif _close(got, (dx + dy) ** 0.5, 1e-9):
        msg.append("You added the differences first. Square each difference, add them,")
        msg.append("then take the square root.")
    elif _close(got, abs(dx) + abs(dy), 1e-9):
        msg.append("That is the distance walked along the grid, not the straight line.")
    else:
        msg.append("Pattern: ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5")
    return FAIL, msg


# ---------------------------------------------------------------- 4.3
@_checkpoint("4.3", "Task 4.3 - scholarship eligibility")
def _c43():
    left = _todo("full_award", "partial_award")
    if left:
        return TODO, ["Still to write: " + ", ".join(left) + "."]

    full, part = _v("full_award"), _v("partial_award")
    gpa, units, prob = _v("gpa"), _v("units"), _v("on_probation")
    if not (_num(gpa) and _num(units)):
        gpa, units, prob = 3.6, 9, False
    min_gpa = _v("MIN_GPA") if _num(_v("MIN_GPA")) else 3.5
    min_units = _v("MIN_UNITS") if _num(_v("MIN_UNITS")) else 12

    want_full = gpa >= min_gpa and units >= min_units and not prob
    want_part = gpa >= min_gpa or units >= min_units

    msg = []
    for label, got, want in (("full_award", full, want_full),
                             ("partial_award", part, want_part)):
        if not isinstance(got, bool):
            msg.append("%s is %r, which is not True or False." % (label, got))
            msg.append("   Build it out of comparisons, e.g. gpa >= MIN_GPA.")
        elif got != want:
            msg.append("%s is %s but should be %s." % (label, got, want))

    if not msg:
        return PASS, ["full_award=%s, partial_award=%s - both expressions are right."
                      % (full, part)]

    if isinstance(full, bool) and full != want_full and want_full is False:
        msg.append("For the FULL award all three conditions must hold at once, so they")
        msg.append("are joined with and. units is %s, below the %s required."
                   % (units, min_units))
    if isinstance(part, bool) and part != want_part and want_part is True:
        msg.append("For the PARTIAL award only one condition has to hold, so join them")
        msg.append("with or. gpa is %s, which already meets %s." % (gpa, min_gpa))
    return FAIL, msg


# ---------------------------------------------------------------- 4.4
@_checkpoint("4.4", "Task 4.4 - even, odd and divisible")
def _c44():
    names = ("is_even", "is_divisible_by_3", "last_digit_is_6")
    left = _todo(*names)
    if left:
        return TODO, ["Still to write: " + ", ".join(left) + "."]

    n = _v("number")
    if not isinstance(n, int):
        n = 246
    wants = (n % 2 == 0, n % 3 == 0, n % 10 == 6)
    hints = ("number % 2 == 0", "number % 3 == 0", "number % 10 == 6")

    msg = []
    for label, got, want, hint in zip(names, [_v(x) for x in names], wants, hints):
        if not isinstance(got, bool):
            msg.append("%s is %r, not True or False." % (label, got))
            msg.append("   % gives you the REMAINDER, which is a number. Compare it with")
            msg.append("   == to turn it into a Boolean:  %s" % hint)
        elif got != want:
            msg.append("%s is %s but for number = %s it should be %s   (%s)"
                       % (label, got, n, want, hint))
    if msg:
        return FAIL, msg
    return PASS, ["All three Booleans are correct for number = %s." % n]


# ---------------------------------------------------------------- 5.1
@_checkpoint("5.1", "Task 5.1 - flowchart to code (tip calculator)")
def _c51():
    left = _todo("bill_amount", "tip_percent", "tip", "total")
    if left:
        return TODO, ["The flowchart is not implemented yet - missing: "
                      + ", ".join(left) + "."]

    bill, pct, tip, total = (_v("bill_amount"), _v("tip_percent"), _v("tip"), _v("total"))
    strs = [n for n, x in (("bill_amount", bill), ("tip_percent", pct)) if isinstance(x, str)]
    if strs:
        return FAIL, ["These are still text, not numbers: " + ", ".join(strs) + ".",
                      "input() always hands back a string - wrap it in float()."]
    if not all(_num(x) for x in (bill, pct, tip, total)):
        return FAIL, ["bill_amount, tip_percent, tip and total must all be numbers."]

    msg = []
    want_tip = bill * pct / 100
    if not _close(tip, want_tip, 0.005):
        msg.append("tip is %s but bill %s at %s%% is %.2f." % (tip, bill, pct, want_tip))
        if _close(tip, bill * pct, 0.005):
            msg.append("   You multiplied by the percent but forgot to divide by 100.")
        elif _close(tip, bill + pct, 0.005):
            msg.append("   A percentage is a multiplication, not an addition.")
    want_total = bill + tip
    if not _close(total, want_total, 0.005):
        msg.append("total is %s but bill + tip is %.2f." % (total, want_total))
        if _close(total, tip, 0.005):
            msg.append("   total is the bill PLUS the tip, not the tip on its own.")
    if msg:
        return FAIL, msg

    notes = ["bill %.2f + tip %.2f = total %.2f - the arithmetic is right."
             % (bill, tip, total)]
    if not any("input(" in s and "tip" in s for s in _sources()):
        notes.append("Reminder: the flowchart uses two input parallelograms, so both")
        notes.append("values should be read with input(), not typed into the code.")
    return PASS, notes


# ---------------------------------------------------------------- 5.2
def _symbol_kind(text):
    s = str(text).lower()
    if "termin" in s or "start" in s or s.strip().endswith("end") or ": end" in s:
        return "T"
    if ("input" in s or "output" in s or "read" in s or "display" in s
            or "print" in s or "show" in s):
        return "I"
    if "process" in s or "=" in s or "comput" in s or "calc" in s or "set " in s:
        return "P"
    return "?"


@_checkpoint("5.2", "Task 5.2 - code back to flowchart")
def _c52():
    items = _v("flow_5_2")
    if items is MISSING or items is None:
        return TODO, ["flow_5_2 has not been filled in yet."]
    if not isinstance(items, (list, tuple)):
        return FAIL, ["flow_5_2 should be a list of strings, one per symbol."]
    items = [str(x) for x in items if str(x).strip()]
    if len(items) < 4:
        return TODO, ["Only %d symbol(s) listed - the program has more steps than that."
                      % len(items)]

    kinds = [_symbol_kind(x) for x in items]
    unknown = [items[i] for i, k in enumerate(kinds) if k == "?"]
    if unknown:
        return FAIL, ["These lines do not name a symbol type:"] + \
               ["   " + u for u in unknown] + \
               ['Start each line with "Terminator:", "Input/Output:" or "Process:".']

    core = kinds[:]
    had_start = core and core[0] == "T"
    had_end = core and core[-1] == "T"
    if had_start:
        core = core[1:]
    if had_end and core:
        core = core[:-1]
    if core[:1] == ["P"]:          # the SECONDS_PER_MINUTE constant is optional
        core = core[1:]
    shape = "".join(core)

    if shape in ("IIPPPI", "IIPPPII"):
        notes = ["The symbols are in the right order."]
        if not (had_start and had_end):
            notes.append("Tip: a complete flowchart also has Start and End terminators.")
        return PASS, notes

    msg = ["The order of symbols does not match the program yet."]
    ins, procs, outs = core.count("I"), core.count("P"), 0
    first_p = core.index("P") if "P" in core else -1
    if first_p >= 0:
        ins = core[:first_p].count("I")
        outs = core[first_p:].count("I")
    if ins != 2:
        msg.append("The program calls input() twice (distance_km and minutes), so there")
        msg.append("should be 2 input parallelograms before any processing - you have %d." % ins)
    if procs != 3:
        msg.append("It computes three values (seconds, speed_kmh, pace_per_km), so there")
        msg.append("should be 3 process rectangles - you have %d." % procs)
    if outs not in (1, 2):
        msg.append("It prints two lines at the end, so there should be 1 or 2 output")
        msg.append("parallelograms after the processing - you have %d." % outs)
    if ins == 2 and procs == 3 and outs in (1, 2):
        msg.append("All the pieces are there but not in program order: read both inputs")
        msg.append("first, then compute, then display.")
    return FAIL, msg


# ---------------------------------------------------------------- 6.1
@_checkpoint("6.1", "Task 6.1 - debugging Celsius to Fahrenheit")
def _c61():
    cel, fah = _v("celsius"), _v("fahrenheit")
    if cel is MISSING or fah is MISSING:
        return TODO, ["The Task 6.1 cell has not been run yet."]
    if isinstance(cel, str):
        return FAIL, ["celsius is the string %r, not a number." % cel,
                      "Bug 1: input() always returns text. Convert it: float(input(...))."]
    if not _num(cel):
        return FAIL, ["celsius should be a number, not %r." % (cel,)]
    if not _num(fah):
        return FAIL, ["fahrenheit should be a number, not %r." % (fah,)]

    want = cel * 9 / 5 + 32
    if not _close(fah, want, 1e-9):
        msg = ["For %s C the answer is %s F, but fahrenheit is %s." % (cel, want, fah)]
        if _close(fah, cel * (9 // 5) + 32, 1e-9):
            msg.append("Bug 2: 9 // 5 is floor division and gives 1, so you are using a")
            msg.append("factor of 1 instead of 1.8. Use 9 / 5.")
        elif _close(fah, cel * 9 / 5, 1e-9):
            msg.append("The + 32 is missing.")
        elif _close(fah, (cel + 32) * 9 / 5, 1e-9):
            msg.append("Multiply first, then add 32.")
        else:
            msg.append("The formula is  celsius * 9 / 5 + 32.")
        return FAIL, msg

    if abs(cel - 100) > 1e-9:
        return FAIL, ["The formula is correct (%s C = %s F)." % (cel, fah),
                      "Now run the cell once more and type 100 when prompted, so the",
                      "required line 100.0 C = 212.0 F is the one shown in your notebook."]

    notes = ["100.0 C = 212.0 F - all three bugs fixed."]
    recent = [s for s in _sources() if "Celsius" in s]
    if recent and not re.search(r"\{[^}]*:\s*\.\d+f\}", recent[-1]):
        notes.append("Bug 3 note: your values print correctly, but add a format specifier")
        notes.append("such as {celsius:.1f} so the decimals never depend on the input.")
    return PASS, notes


# =====================================================================
#  Display helpers
# =====================================================================
try:
    from IPython.display import HTML, display
    _RICH = True
except Exception:
    _RICH = False

_STYLE = {
    PASS: ("#15803d", "#dcfce7", "#22c55e", "PASSED"),
    FAIL: ("#b91c1c", "#fee2e2", "#ef4444", "NOT YET"),
    TODO: ("#a16207", "#fef3c7", "#f59e0b", "TO DO"),
}


def _esc(t):
    return (str(t).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))


def _run(cid):
    """Grade one checkpoint, never raising."""
    title, fn = CHECKS[cid]
    try:
        status, lines = fn()
    except Exception as exc:
        status = FAIL
        lines = ["The check could not run: %s: %s" % (type(exc).__name__, exc),
                 "This usually means an earlier cell was not run. Try",
                 "Runtime -> Restart and run all."]
    if isinstance(lines, str):
        lines = [lines]
    return status, [str(x) for x in lines]


def report(cid):
    """Run a single checkpoint and show the result."""
    title, _ = CHECKS[cid]
    status, lines = _run(cid)
    fg, bg, dot, word = _STYLE[status]

    if _RICH:
        body = "<br>".join(_esc(l) for l in lines)
        display(HTML(
            "<div style='font-family:ui-monospace,SFMono-Regular,Menlo,monospace;"
            "background:%s;color:#111827;border-left:6px solid %s;border-radius:8px;"
            "padding:10px 14px;margin:4px 0;max-width:820px;'>"
            "<div style='font-weight:700;color:%s;font-size:13px;letter-spacing:.4px;'>"
            "%s &nbsp;CHECK %s &mdash; %s</div>"
            "<div style='font-size:13px;line-height:1.55;margin-top:4px;"
            "white-space:pre-wrap;'>%s</div></div>"
            % (bg, dot, fg, word, _esc(cid), _esc(title), body)))
    else:
        print("[%s] CHECK %s - %s" % (word, cid, title))
        for line in lines:
            print("   " + line)
    return status


def final_check():
    """Grade every checkpoint and show the submission dashboard."""
    results = [(cid, CHECKS[cid][0]) + _run(cid) for cid in ORDER]
    total = len(results)
    passed = sum(1 for r in results if r[2] == PASS)
    ready = passed == total

    name = _v("student_name")
    filename = None
    if isinstance(name, str) and "your name" not in name.lower():
        parts = [p for p in re.split(r"[^A-Za-z'\-]+", name) if p]
        if len(parts) >= 2:
            filename = "L03_%s_%s" % (parts[0], parts[-1])

    if not _RICH:
        print("=" * 62)
        print(" CS 250 Lab 3 - final check:  %d of %d checkpoints passed" % (passed, total))
        print("=" * 62)
        for cid, title, status, lines in results:
            print(" %-8s %-46s %s" % (cid, title, _STYLE[status][3]))
            if status != PASS:
                print("          -> " + lines[0])
        print("-" * 62)
        if ready:
            print(" ALL CHECKPOINTS PASSED - you are ready to submit.")
            print(" 1. Runtime -> Restart and run all, one last time.")
            print(" 2. Name the notebook file %s.ipynb"
                  % (filename or "L03_firstname_lastname"))
            print(" 3. Upload/push it to your GitHub repository.")
            print(" 4. Copy the GitHub URL of the notebook.")
            print(" 5. Paste that URL into the Lab 3 assignment in Canvas.")
        else:
            print(" Not ready yet - fix the checkpoints marked above, then run this cell again.")
        return ready

    lights, rows = [], []
    for cid, title, status, lines in results:
        fg, bg, dot, word = _STYLE[status]
        glow = ("box-shadow:0 0 0 4px rgba(34,197,94,.25);" if status == PASS else "")
        lights.append(
            "<div style='text-align:center;min-width:52px;'>"
            "<div style='width:20px;height:20px;border-radius:50%%;background:%s;"
            "margin:0 auto 5px;%s'></div>"
            "<div style='font-size:11px;color:#6b7280;'>%s</div></div>"
            % (dot, glow, _esc(cid)))
        detail = ""
        if status != PASS:
            detail = ("<div style='font-size:12px;color:#4b5563;margin-top:3px;"
                      "white-space:pre-wrap;'>%s</div>"
                      % "<br>".join(_esc(l) for l in lines[:3]))
        rows.append(
            "<tr><td style='padding:9px 10px;border-top:1px solid #e5e7eb;"
            "vertical-align:top;width:26px;'>"
            "<div style='width:13px;height:13px;border-radius:50%%;background:%s;'></div></td>"
            "<td style='padding:9px 10px;border-top:1px solid #e5e7eb;vertical-align:top;'>"
            "<span style='font-weight:600;color:#111827;font-size:13px;'>%s</span>"
            "<span style='color:#6b7280;font-size:12px;'> &nbsp;%s</span>%s</td>"
            "<td style='padding:9px 10px;border-top:1px solid #e5e7eb;vertical-align:top;"
            "text-align:right;white-space:nowrap;'>"
            "<span style='background:%s;color:%s;font-size:11px;font-weight:700;"
            "padding:3px 9px;border-radius:20px;letter-spacing:.3px;'>%s</span></td></tr>"
            % (dot, _esc(cid), _esc(title), detail, bg, fg, word))

    pct = int(round(100 * passed / total))
    bar_colour = "#22c55e" if ready else "#f59e0b"

    if ready:
        banner = (
            "<div style='background:#dcfce7;border:1px solid #86efac;border-radius:12px;"
            "padding:18px 20px;margin-top:18px;'>"
            "<div style='font-size:18px;font-weight:800;color:#15803d;'>"
            "&#10003; ALL %d CHECKPOINTS PASSED &mdash; YOU ARE READY TO SUBMIT</div>"
            "<div style='color:#166534;font-size:13.5px;line-height:1.85;margin-top:10px;'>"
            "<b>1.</b> Run <b>Runtime &rarr; Restart and run all</b> one last time and make "
            "sure every cell still finishes.<br>"
            "<b>2.</b> Name your notebook file "
            "<code style='background:#ffffff;border:1px solid #86efac;border-radius:5px;"
            "padding:2px 7px;font-weight:700;'>%s.ipynb</code><br>"
            "<b>3.</b> Upload or push that file to your <b>GitHub</b> repository.<br>"
            "<b>4.</b> Open it on GitHub and copy the <b>URL of the notebook</b>.<br>"
            "<b>5.</b> Paste that URL into the <b>Lab 3</b> assignment in <b>Canvas</b>."
            "</div>"
            "<div style='margin-top:12px;font-size:12.5px;color:#166534;'>"
            "The filename must follow <b>L03_firstname_lastname</b> exactly &mdash; "
            "submissions that are named differently cannot be matched to you in Canvas."
            "</div></div>"
            % (total, _esc(filename or "L03_firstname_lastname")))
    else:
        pending = [r for r in results if r[2] != PASS]
        items = "".join("<li><b>%s</b> %s &mdash; %s</li>"
                        % (_esc(cid), _esc(title), _esc(lines[0]))
                        for cid, title, status, lines in pending)
        banner = (
            "<div style='background:#fef3c7;border:1px solid #fcd34d;border-radius:12px;"
            "padding:18px 20px;margin-top:18px;'>"
            "<div style='font-size:17px;font-weight:800;color:#a16207;'>"
            "NOT READY TO SUBMIT &mdash; %d checkpoint(s) still need attention</div>"
            "<ul style='color:#78350f;font-size:13px;line-height:1.7;margin:10px 0 0 0;"
            "padding-left:20px;'>%s</ul>"
            "<div style='margin-top:12px;font-size:13px;color:#78350f;'>"
            "Go back to each task above, read the message in its CHECK cell, fix the code, "
            "re-run the cell, then run this final cell again.</div></div>"
            % (len(pending), items))

    display(HTML(
        "<div style='font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,"
        "Helvetica,Arial,sans-serif;background:#ffffff;color:#111827;border:1px solid "
        "#e5e7eb;border-radius:16px;padding:22px 24px;max-width:860px;"
        "box-shadow:0 1px 3px rgba(0,0,0,.08);'>"
        "<div style='font-size:20px;font-weight:800;'>CS 250 &middot; Lab 3 "
        "&middot; Submission check</div>"
        "<div style='color:#6b7280;font-size:13px;margin-top:3px;'>%s &nbsp;&middot;&nbsp; "
        "<b style='color:%s;'>%d of %d</b> checkpoints passed</div>"
        "<div style='background:#f3f4f6;border-radius:99px;height:9px;margin:14px 0 18px;'>"
        "<div style='background:%s;height:9px;border-radius:99px;width:%d%%;'></div></div>"
        "<div style='display:flex;flex-wrap:wrap;gap:6px;justify-content:flex-start;"
        "margin-bottom:16px;'>%s</div>"
        "<table style='width:100%%;border-collapse:collapse;'>%s</table>%s"
        "<div style='color:#9ca3af;font-size:11.5px;margin-top:14px;'>"
        "This dashboard re-checks your variables every time it runs, so always use "
        "Restart and run all before you trust it.</div></div>"
        % (_esc(name if isinstance(name, str) else "unnamed student"),
           bar_colour, passed, total, bar_colour, pct,
           "".join(lights), "".join(rows), banner)))
    return ready


print("Checkpoint helpers loaded: %d checkpoints registered." % len(CHECKS))
print("Run the CHECK cell after each task, and final_check() at the end.")

Checkpoint helpers loaded: 11 checkpoints registered.
Run the CHECK cell after each task, and final_check() at the end.


---
### Part 0.1 · Identify yourself

In [2]:
# TODO: replace with your own details
student_name = "Yi Cui"     # use your first AND last name
lab_date = "2026_08_27"

print(f"CS 250 Lab 3 — {student_name} — {lab_date}")

CS 250 Lab 3 — Yi Cui — 2026_08_27


#### ✅ CHECK 0.1 — your details

In [3]:
report("0.1")

'pass'

---
## Part 1 · Arithmetic operators

| Operator | Name | `7` and `3` |
|---|---|---|
| `+` | addition | `10` |
| `-` | subtraction | `4` |
| `*` | multiplication | `21` |
| `/` | true division — **always a float** | `2.333...` |
| `//` | floor division — rounds **down** | `2` |
| `%` | modulo — the remainder | `1` |
| `**` | exponentiation | `343` |

### GUIDED EXAMPLE 1.1 — the three divisions

Run the cell. Pay attention to the negative numbers: `//` rounds **down** toward
negative infinity, not toward zero.

In [4]:
print(17 / 5,  17 // 5,  17 % 5)     # 3.4   3   2
print(10 / 2,  10 // 2,  10 % 2)     # 5.0   5   0
print(-7 / 2,  -7 // 2,  -7 % 2)     # -3.5  -4  1   <- surprising!

# The identity that always holds:
a, b = -7, 2
print(a == (a // b) * b + (a % b))   # True

3.4 3 2
5.0 5 0
-3.5 -4 1
True


### GUIDED EXAMPLE 1.2 — where `//` and `%` earn their keep

Splitting a total into whole units and a remainder is the classic use.

In [5]:
total_seconds = 3725

hours   = total_seconds // 3600
minutes = (total_seconds % 3600) // 60
seconds = total_seconds % 60

print(f"{total_seconds} seconds = {hours}h {minutes}m {seconds}s")

3725 seconds = 1h 2m 5s


### TASK 1.3 — make change

A customer pays with a number of **cents**. Break the amount into quarters (25c),
dimes (10c), nickels (5c) and pennies (1c), always using the largest coins first.

For `total_cents = 287` the answer is **11 quarters, 1 dime, 0 nickels, 2 pennies**.

Use only `//` and `%` — no `if` statements (we meet those next week).

In [6]:
total_cents = 287

# TODO: compute each coin count
quarters  = total_cents // 25
remainder = total_cents % 25
dimes     = (total_cents % 25) // 10
remainder = (total_cents % 25) % 10
nickels   = ((total_cents % 25) % 10) // 5
pennies   = ((total_cents % 25) % 10) % 5

print(f"{quarters} quarters, {dimes} dimes, {nickels} nickels, {pennies} pennies")

11 quarters, 1 dimes, 0 nickels, 2 pennies


#### ✅ CHECK 1.3 — make change

In [7]:
report("1.3")

'pass'

---
## Part 2 · Operator precedence

Python evaluates operators in a fixed order, **not** left to right:

```
( )   ->   **   ->   -x (unary)   ->   *  /  //  %   ->   +  -
```

`**` is *right*-associative: `2 ** 3 ** 2` is `2 ** 9`, not `(2 ** 3) ** 2`.

### TASK 2.1 — predict before you run

Write each prediction as a number in the variables below. Then run the CHECK cell,
and only after that run Task 2.2 to see the real answers.

| # | Expression |
|---|---|
| 1 | `2 + 3 * 4` |
| 2 | `(2 + 3) * 4` |
| 3 | `10 - 2 - 3` |
| 4 | `2 ** 3 ** 2` |
| 5 | `20 // 3 * 3` |
| 6 | `7 + 15 % 4` |

In [8]:
# TODO: your predictions as numbers
p1 = 14
p2 = 20
p3 = 5
p4 = 512
p5 = 18
p6 = 10


#### ✅ CHECK 2.1 — precedence predictions

In [9]:
report("2.1")

'pass'

### TASK 2.2 — rewrite for clarity

The expression below computes an average, but it is wrong because of precedence.

```python
average = a + b + c / 3
```

Fix it with parentheses, then use it to average the three quiz scores.

In [10]:
a, b, c = 88, 92, 79

# TODO: compute the correct average
average = (a + b + c)/3

if average is not None:
    print(f"Average score: {average:.2f}")   # should print 86.33

Average score: 86.33


#### ✅ CHECK 2.2 — rewrite for clarity

In [11]:
report("2.2")

'pass'

---
## Part 3 · Translating formulas into Python  

Three rules:
1. Multiplication is never implicit: `4x` becomes `4 * x`.
2. A fraction bar is a big parenthesis: wrap the whole numerator **and** the whole denominator.
3. A superscript becomes `**`; a square root becomes `** 0.5`.

### GUIDED EXAMPLE 3.1 — area and circumference of a circle

In [12]:
PI = 3.14159
radius = 7.5

area          = PI * radius ** 2
circumference = 2 * PI * radius

print(f"Area:          {area:.2f}")
print(f"Circumference: {circumference:.2f}")

Area:          176.71
Circumference: 47.12


### TASK 3.2 — Body Mass Index

The BMI formula is:

```
             weight_kg
BMI  =  ---------------------
          height_m x height_m
```

Compute it for the values given. Correct answer for 68 kg and 1.72 m: **22.99**.

In [13]:
weight_kg = 68.0
height_m  = 1.72

# TODO: translate the formula
bmi = weight_kg / (height_m * height_m)

if bmi is not None:
    print(f"BMI: {bmi:.2f}")

BMI: 22.99


#### ✅ CHECK 3.2 — body mass index

In [14]:
report("3.2")

'pass'

### TASK 3.3 — the distance between two points

```
distance = square root of ( (x2 - x1) squared + (y2 - y1) squared )
```

For (1, 2) and (4, 6) the distance is exactly **5.0**. Remember: square root is `** 0.5`.

In [15]:
x1, y1 = 1, 2
x2, y2 = 4, 6

# TODO: translate the formula
distance = ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5

print(f"Distance: {distance}")

Distance: 5.0


#### ✅ CHECK 3.3 — distance between two points

In [16]:
report("3.3")

'pass'

---
## Part 4 · Booleans, comparisons and logic

A comparison is an expression whose value is `True` or `False`.

| | |
|---|---|
| `==` equal to | `!=` not equal to |
| `<` less than | `>` greater than |
| `<=` at most | `>=` at least |

Combine them with `and`, `or`, `not`. Precedence: `not` first, then `and`, then `or`.

### GUIDED EXAMPLE 4.1 — comparisons produce Booleans

In [17]:
score = 87

passed = score >= 70
print(passed, type(passed))

print(0 <= score <= 100)     # chained comparison
print(5 == "5")              # False - different types
print(5 == 5.0)              # True  - same numeric value
print(0.1 + 0.2 == 0.3)      # False - floats are approximate!

True <class 'bool'>
True
False
True
False


### GUIDED EXAMPLE 4.2 — truthiness

`bool()` turns any value into `True` or `False`. Empty or zero is falsy; everything else is truthy.

In [18]:
for value in [0, 0.0, "", " ", "0", "False", 42, -1, None]:
    print(f"{repr(value):>10}  ->  {bool(value)}")

         0  ->  False
       0.0  ->  False
        ''  ->  False
       ' '  ->  True
       '0'  ->  True
   'False'  ->  True
        42  ->  True
        -1  ->  True
      None  ->  False


### TASK 4.3 — scholarship eligibility

A student qualifies for the scholarship when **all three** are true:
- GPA is at least 3.5
- they are taking at least 12 units
- they are **not** on academic probation

They qualify for the *partial* award when they meet the GPA requirement **or** the units
requirement (at least one), regardless of probation.

Compute both Booleans below. Do not use `if` — just build the expressions.

In [19]:
gpa          = 3.6
units        = 9
on_probation = False

MIN_GPA   = 3.5
MIN_UNITS = 12

# TODO: build the two Boolean expressions
full_award    = (gpa >= MIN_GPA) and (units >= MIN_UNITS) and (on_probation == False)
partial_award = gpa >= MIN_GPA or units >= MIN_UNITS

print(f"Full award:    {full_award}")      # expected False
print(f"Partial award: {partial_award}")   # expected True

Full award:    False
Partial award: True


#### ✅ CHECK 4.3 — scholarship eligibility

In [20]:
report("4.3")

'pass'

### TASK 4.4 — even, odd and divisible

Using `%` and `==`, produce Booleans that answer three questions about `number`.
Do not use `if`.

In [21]:
number = 246

# TODO
is_even            = number % 2 == 0    # True when number divides evenly by 2
is_divisible_by_3  = number % 3 ==0    # True when number divides evenly by 3
last_digit_is_6    = number % 10 ==6   # True when the final digit is 6

print(is_even, is_divisible_by_3, last_digit_is_6)   # expected: True True True

True True True


#### ✅ CHECK 4.4 — even, odd and divisible

In [22]:
report("4.4")

'pass'

---
## Part 5 · From flowchart to code

A flowchart is a picture of an algorithm. Each symbol maps onto Python:

| Symbol | Meaning | Python |
|---|---|---|
| Rounded box | Start / End | a comment |
| Parallelogram | Input / Output | `input()` / `print()` |
| Rectangle | Process | an assignment |
| Diamond | Decision | `if` (Week 3) |

### TASK 5.1 — implement this flowchart

```
              +-------------+
              |    START    |
              +------+------+
                     |
          +----------v-----------+
         /  Read  bill_amount    /
        +-----------+-----------+
                     |
          +----------v-----------+
         /  Read  tip_percent    /
        +-----------+-----------+
                     |
        +------------v-------------+
        |  tip = bill_amount       |
        |        x tip_percent/100 |
        +------------+-------------+
                     |
        +------------v-------------+
        |  total = bill_amount+tip |
        +------------+-------------+
                     |
        +------------v-------------+
       /  Display tip and total    /
      +-------------+--------------+
                    |
             +------v------+
             |     END     |
             +-------------+
```

Write the program below. Use the `# --- INPUT ---` / `# --- PROCESS ---` / `# --- OUTPUT ---`
structure, and show both money values to two decimals.

In [23]:
# Use exactly these variable names so CHECK 5.1 can see your work:
#   bill_amount, tip_percent, tip, total

# --- INPUT ---
# TODO
bill_amount = float(input("bill_amount: "))
tip_percent = float(input("tip_percent: "))
# --- PROCESS ---
# TODO
tip = bill_amount * tip_percent / 100
total = bill_amount + tip
# --- OUTPUT ---
# TODO
print(f"tip: ${tip:.2f}")
print(f"total: ${total:.2f}")


bill_amount: 100
tip_percent: 15
tip: $15.00
total: $115.00


#### ✅ CHECK 5.1 — flowchart to code

In [24]:
report("5.1")

'pass'

### TASK 5.2 — now go the other way

Read the program below, then describe its flowchart by listing the symbols in order.

Write the description twice: once as prose in the markdown cell, and once as a Python
list in the code cell after it, so that CHECK 5.2 can look at it. Use this format, one
line per symbol:

```
1. Terminator: Start
2. Input/Output: read ...
3. Process: ...
```

In [25]:
# Read this program - do not change it. Run it once to see what it does.
SECONDS_PER_MINUTE = 60

distance_km = float(input("Distance in km: "))
minutes     = float(input("Time in minutes: "))

seconds     = minutes * SECONDS_PER_MINUTE
speed_kmh   = distance_km / (minutes / 60)
pace_per_km = seconds / distance_km

print(f"Speed: {speed_kmh:.2f} km/h")
print(f"Pace:  {pace_per_km:.1f} seconds per km")

Distance in km: 60
Time in minutes: 60
Speed: 60.00 km/h
Pace:  60.0 seconds per km


**Your flowchart description for Task 5.2 — replace this text:**

1. Terminator start
2. Input/Output: read Distance in km
3. Input/Output: read Time in minutes
4. Process: seconds     = minutes * SECONDS_PER_MINUTE
5. Process: speed_kmh   = distance_km / (minutes / 60)
6. Process: pace_per_km = seconds / distance_km
7. display speed and pace
8. end

In [26]:
# TODO: one string per flowchart symbol, in the order the program runs.
# Start each string with "Terminator:", "Input/Output:" or "Process:".
flow_5_2 = [
"Terminator: Start",
"Input/Output: read distance_km",
"Input/Output: read minutes",
"Process: seconds = minutes * SECONDS_PER_MINUTE",
"Process: speed_kmh = distance_km / (minutes / 60)",
"Process: pace_per_km = seconds / distance_km",
"Input/Output: display: speed_kmh",
"Input/Output: display: pace_per_km",
"Terminator: end"
]

#### ✅ CHECK 5.2 — code back to flowchart

In [27]:
report("5.2")

'pass'

---
## Part 6 · Debugging

### TASK 6.1 — three more bugs

This program should convert a Celsius temperature to Fahrenheit.
With an input of 100 it must print `100.0 C = 212.0 F`. It has **three** bugs.

In [28]:
# BUGGY - fix me
celsius = int(input("celsius: "))

fahrenheit = (9/5) * celsius + 32

print(f"{fahrenheit:.1f} F")


celsius: 100
212.0 F


#### Hint (open only if stuck)

<details>
<summary>Show hints</summary>

1. `input()` returns a `str` — you cannot multiply text by a number and get arithmetic.
2. `9 // 5` is floor division, which gives `1`, not `1.8`. Which operator did you want?
3. The output should show one decimal place on each value — add a format specifier.

</details>

#### ✅ CHECK 6.1 — Celsius to Fahrenheit

> Until you fix the bugs, the cell above **raises a `TypeError`** — that is intentional. Note that **Restart and run all stops at the first error**, so this check and the final dashboard will not run until Task 6.1 works. Fix it, then run everything again.

In [29]:
report("6.1")

'pass'

---
## Before you submit

- [ ] `Runtime → Restart and run all` completes with no errors
- [ ] Your name appears in the Part 0 output
- [ ] Every CHECK cell reports **PASSED**
- [ ] The final dashboard below shows **all green lights**

**Submit:** name the notebook `L03_firstname_lastname.ipynb`, push it to GitHub, and
paste the notebook's GitHub URL into the Lab 3 assignment in Canvas.

---
## Part 7 · Final check — are you ready to submit?

Run the cell below. It re-checks **every** task in the notebook and shows a dashboard.

- **All green** → it will show you your submission instructions and your exact filename.
- **Any amber or red** → it will list what still needs fixing and why. Go back, fix it,
  re-run that task's CHECK cell, then run this cell again.

> Always do **Runtime → Restart and run all** before you trust the dashboard — it reads
> the variables that are currently in memory.

In [30]:
final_check()

,0.1 Part 0 - your details,PASSED
,1.3 Task 1.3 - make change,PASSED
,2.1 Task 2.1 - precedence predictions,PASSED
,2.2 Task 2.2 - rewrite for clarity,PASSED
,3.2 Task 3.2 - body mass index,PASSED
,3.3 Task 3.3 - distance between two points,PASSED
,4.3 Task 4.3 - scholarship eligibility,PASSED
,"4.4 Task 4.4 - even, odd and divisible",PASSED
,5.1 Task 5.1 - flowchart to code (tip calculator),PASSED
,5.2 Task 5.2 - code back to flowchart,PASSED
,6.1 Task 6.1 - debugging Celsius to Fahrenheit,PASSED


True